In [1]:
!pip -q install fastapi uvicorn pyngrok nest_asyncio "transformers>=4.43" "accelerate>=0.33" "bitsandbytes>=0.43"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.1 MB/s eta 0:00:00:00:0100:01


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

base_id = "Qwen/Qwen2.5-7B-Instruct"   # faster than the 12B; also good at Arabic
tok = AutoTokenizer.from_pretrained(base_id, use_fast=True)
bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(base_id, quantization_config=bnb_cfg, device_map={"": 0})
model.eval()

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear4bit(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear4bit(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear4bit(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm

In [3]:
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
from typing import Optional, List, Dict

API_TOKEN = "secret123"        # same value as in the Streamlit sidebar
app = FastAPI()

class Req(BaseModel):
    prompt: Optional[str] = None
    messages: Optional[List[Dict[str, str]]] = None
    max_length: int = 300
    temperature: float = 0.3

@app.post("/generate")
def generate(req: Req, authorization: str = Header(None)):
    if authorization != f"Bearer {API_TOKEN}":
        raise HTTPException(status_code=401, detail="Unauthorized")
    msgs = req.messages or [{"role": "user", "content": req.prompt}]
    inputs = tok.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    n_in = inputs["input_ids"].shape[1]
    kw = dict(do_sample=True, temperature=req.temperature) if req.temperature > 0 else dict(do_sample=False)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=req.max_length,
                             pad_token_id=tok.eos_token_id,
                             stop_strings=["Observation:"], tokenizer=tok, **kw)
    text = tok.decode(out[0][n_in:], skip_special_tokens=True)
    return {"response": text.split("Observation:")[0].strip()}

## 3) Expose with ngrok and serve (keep this cell running)

In [ ]:
import uvicorn
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

ngrok.kill()   
ngrok.set_auth_token(UserSecretsClient().get_secret("NGROK_TOKEN"))
public_url = ngrok.connect(8000).public_url
print("PASTE THIS IN STREAMLIT:", public_url)

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info"))
await server.serve()

INFO:     Started server process [115]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


PASTE THIS IN STREAMLIT: https://prefix-scanner-wrinkle.ngrok-free.dev
INFO:     197.36.212.31:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     197.36.212.31:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     197.36.212.31:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     197.36.212.31:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     197.36.212.31:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     197.36.212.31:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     197.36.212.31:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     197.36.212.31:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     197.36.212.31:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     197.36.212.31:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     197.36.212.31:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     197.36.212.31:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     197.36.212.31:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     197.36.212.31:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     197.36.212.31:0 - "POST /generate HTTP/1.1" 200 OK
INFO:     197.